# 08a — Kaggle Multilingual Spam (French Subset)

**C1 Source Type:** `Fichier de données (CSV / data file)`

---

## Objective

Load and extract the **French subset** from the Kaggle Multilingual Spam dataset
(`rajnathpatel/multilingual-spam-data`). This dataset contains SMS messages
translated into 4 languages (EN, HI, DE, FR) with ham/spam labels.

### Dataset Info

| Field | Value |
|-------|-------|
| Source | [Kaggle: rajnathpatel/multilingual-spam-data](https://www.kaggle.com/datasets/rajnathpatel/multilingual-spam-data) |
| Format | CSV file (downloaded via Kaggle CLI) |
| Rows | ~5,574 |
| Columns | `labels`, `text` (EN), `text_hi`, `text_de`, `text_fr` |
| Labels | ham / spam |
| French | Pre-translated in `text_fr` column |

### Pipeline

```
CSV file → Load with pandas → Extract text_fr column → Normalize schema → Export
```

### Output

- `data/raw/csv/fr/kaggle_multilingual_fr_<N>_<date>.csv`

In [1]:
# ── Imports & Constants ──────────────────────────────────────────────
from __future__ import annotations

import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# ── Configuration ────────────────────────────────────────────────────
CSV_PATH: Path = Path("data/raw/csv/kaggle_multilingual_spam.csv")
OUTPUT_DIR: Path = Path("data/raw/csv/fr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_TEXT_LENGTH: int = 20
SCHEMA_COLS: list[str] = ["text", "label", "source", "language"]

print(f"Input file   : {CSV_PATH}")
print(f"Output dir   : {OUTPUT_DIR.resolve()}")
print(f"Min length   : {MIN_TEXT_LENGTH}")

Input file   : data/raw/csv/kaggle_multilingual_spam.csv
Output dir   : /Users/michaeladebayo/Documents/Simplon/brief_projects/sicurre/data/raw/csv/fr
Min length   : 20


## 1. Load CSV

In [2]:
# ── Load the Kaggle CSV ──────────────────────────────────────────────
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Kaggle CSV not found at {CSV_PATH}. "
        "Download via: kaggle datasets download rajnathpatel/multilingual-spam-data"
    )

df_raw: pd.DataFrame = pd.read_csv(CSV_PATH)
print(f"Shape    : {df_raw.shape}")
print(f"Columns  : {list(df_raw.columns)}")
print(f"\nLabel distribution:")
print(df_raw["labels"].value_counts())
df_raw.head(3)

Shape    : (5572, 5)
Columns  : ['labels', 'text', 'text_hi', 'text_de', 'text_fr']

Label distribution:
labels
ham     4825
spam     747
Name: count, dtype: int64


,labels,text,text_hi,text_de,text_fr
0,ham,"Go until jurong point, crazy.. Available only ...","Dakag बिंदु तक जाओ, पागल. केवल Bag Non महान वि...","Gehen Sie bis jurong Punkt, verrückt.. Verfügb...","Allez jusqu'à Jurong point, fou.. Disponible s..."
1,ham,Ok lar... Joking wif u oni...,ओके लामर.... if if uue पर.,Ok Lar... joking wif u oni...,J'ai fait une blague sur le wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Fktatatat 21 मई को प्राप्त करने के लिए मुफ्त प...,Freier Eintritt in 2 a wkly comp zum Gewinn FA...,Entrée libre dans 2 a wkly comp pour gagner FA...


## 2. Extract French Subset

The `text_fr` column contains pre-translated French versions of each SMS.
We extract this column and normalize to the unified schema.

In [3]:
# ── Extract French text and normalize ────────────────────────────────
df_fr = pd.DataFrame({
    "text": df_raw["text_fr"].astype(str),
    "label": df_raw["labels"].apply(
        lambda x: "spam" if str(x).lower() == "spam" else "ham"
    ),
    "source": "kaggle_multilingual_spam",
    "language": "fr",
})

# Filter short / empty texts
before: int = len(df_fr)
df_fr = df_fr[df_fr["text"].str.len() >= MIN_TEXT_LENGTH].reset_index(drop=True)
print(f"Before filter : {before:,}")
print(f"After filter  : {len(df_fr):,} (removed {before - len(df_fr)} short texts)")
print(f"\nLabel distribution:")
print(df_fr["label"].value_counts())
print(f"\nSample:")
df_fr.head(5)

Before filter : 5,572
After filter  : 5,367 (removed 205 short texts)

Label distribution:
label
ham     4625
spam     742
Name: count, dtype: int64

Sample:


,text,label,source,language
0,"Allez jusqu'à Jurong point, fou.. Disponible s...",ham,kaggle_multilingual_spam,fr
1,J'ai fait une blague sur le wif u oni...,ham,kaggle_multilingual_spam,fr
2,Entrée libre dans 2 a wkly comp pour gagner FA...,spam,kaggle_multilingual_spam,fr
3,U dun dit si tôt hor... U c déjà dire alors...,ham,kaggle_multilingual_spam,fr
4,"Non, je ne pense pas qu'il va à usf, il vit da...",ham,kaggle_multilingual_spam,fr


## 3. Deduplicate

In [4]:
# ── Deduplication by text hash ────────────────────────────────────────
before = len(df_fr)
df_fr["text_hash"] = df_fr["text"].str[:300].apply(
    lambda t: hashlib.sha256(t.encode("utf-8", errors="ignore")).hexdigest()
)
df_fr = (
    df_fr.drop_duplicates(subset="text_hash", keep="first")
    .drop(columns="text_hash")
    .reset_index(drop=True)
)
after: int = len(df_fr)
print(f"Before dedup : {before:,}")
print(f"After dedup  : {after:,}")
print(f"Removed      : {before - after:,}")

Before dedup : 5,367
After dedup  : 4,981
Removed      : 386


## 4. Export

In [5]:
# ── Export ─────────────────────────────────────────────────────────────
timestamp: str = datetime.now(timezone.utc).strftime("%Y%m%d")
n_rows: int = len(df_fr)

if n_rows > 0:
    filename: str = f"kaggle_multilingual_fr_{n_rows}_{timestamp}.csv"
    output_path: Path = OUTPUT_DIR / filename
    df_fr.to_csv(output_path, index=False, encoding="utf-8")

    size_kb: float = output_path.stat().st_size / 1024
    print(f"Exported  : {output_path}")
    print(f"Rows      : {n_rows:,}")
    print(f"Size      : {size_kb:.1f} KB")
    print(f"Columns   : {list(df_fr.columns)}")
else:
    print("Nothing to export.")

Exported  : data/raw/csv/fr/kaggle_multilingual_fr_4981_20260301.csv
Rows      : 4,981
Size      : 608.6 KB
Columns   : ['text', 'label', 'source', 'language']


## 5. Summary

| Criterion | Evidence |
|-----------|----------|
| **Source type** | Fichier de données — CSV downloaded from Kaggle |
| **Provider** | `rajnathpatel/multilingual-spam-data` |
| **French content** | Pre-translated `text_fr` column |
| **Schema** | Normalized to `(text, label, source, language)` |
| **Deduplication** | SHA-256 hash of first 300 chars |